# 21 · BirdNET vs. ResNet152V2 — Evaluación pareada y comparación

Segunda parte de la comparación. Aquí se **re-evalúa el ResNet152V2 a nivel de grabación**
(recording-level) sobre **exactamente las mismas grabaciones** que BirdNET predijo en el
notebook `20_birdnet_comparison.ipynb`, y se comparan ambos modelos de forma pareada.

El ResNet trabaja sobre los espectrogramas JPEG (que sí se conservan), agregando las
predicciones de todos los chunks de 5 s de cada grabación (media de probabilidades → top-1).

> **Kernel**: ejecuta este notebook con el **`.venv` del proyecto** (TensorFlow 2.19), NO con
> `.venv-birdnet`. Requiere haber ejecutado antes el notebook 20 (genera
> `src/data/birdnet_predictions.csv`).

Salidas: `src/data/resnet_predictions.csv`, `src/data/birdnet_vs_resnet_summary.csv`,
`src/data/birdnet_vs_resnet_by_family.csv`, y figuras en `fig/`.

## Nota metodológica

- La comparación es **recording-level top-1** sobre la intersección de grabaciones que ambos
  modelos pudieron predecir (BirdNET necesitó descargar el audio; el ResNet usa los JPEG).
- El baseline publicado del proyecto (accuracy 0.861) es **por chunk sobre todo el test**; los
  números recording-level sobre la muestra pueden diferir (suelen ser algo mejores al agregar
  evidencia de varios chunks). Es la comparación justa frente a BirdNET.
- El `Dropout` del modelo se creó con `training=True` (Monte Carlo dropout), por lo que
  `predict` es ligeramente estocástico; promediar los chunks de cada grabación lo estabiliza.

In [1]:
import os, sys
import numpy as np
import pandas as pd

sys.path.append("../src")
import birdnet_utils as bu
import viz_style as vs
vs.apply_style()   # paleta y rcParams estándar del proyecto

ROOT        = os.path.abspath("..")
IMAGES_ROOT = os.path.join(ROOT, "src/data/images_test/images_spectograms")
DATA_DIR    = os.path.join(ROOT, "src/data")
FIG_DIR     = os.path.join(ROOT, "fig")
MODELS_DIR  = os.path.join(ROOT, "notebooks/models")
WEIGHTS     = os.path.join(MODELS_DIR, "weights_ResNet152V2.weights.h5")
LE_PATH     = os.path.join(MODELS_DIR, "label_encoder_ResNet152V2.pkl")
os.makedirs(FIG_DIR, exist_ok=True)

# Predicciones de BirdNET (del notebook 20)
bird = pd.read_csv(os.path.join(DATA_DIR, "birdnet_predictions.csv"))
bird["recording_id"] = bird.recording_id.astype(str)
bird_valid = bird.dropna(subset=["species_pred"]).copy()
eval_recs = set(bird_valid.recording_id)
print("grabaciones con predicción de BirdNET:", len(eval_recs))

grabaciones con predicción de BirdNET: 11695


In [2]:
# Construir el DataFrame de chunks (un espectrograma por fila) SOLO de las grabaciones a evaluar
index = bu.build_test_index(IMAGES_ROOT)
index["recording_id"] = index.recording_id.astype(str)
sub = index[index.recording_id.isin(eval_recs)]

rows = []
for r in sub.itertuples():
    for p in r.chunk_paths:
        rows.append({"path_img": p, "label": r.species, "recording_id": r.recording_id})
chunks_df = pd.DataFrame(rows).reset_index(drop=True)
print("grabaciones a evaluar:", chunks_df.recording_id.nunique(),
      "| chunks:", len(chunks_df))
chunks_df.head()

grabaciones a evaluar: 11695 | chunks: 21238


,path_img,label,recording_id
0,/Users/camcortes/Documents/birds-sounds/src/da...,Acropternis orthonyx,121142
1,/Users/camcortes/Documents/birds-sounds/src/da...,Acropternis orthonyx,121142
2,/Users/camcortes/Documents/birds-sounds/src/da...,Acropternis orthonyx,121142
3,/Users/camcortes/Documents/birds-sounds/src/da...,Acropternis orthonyx,127718
4,/Users/camcortes/Documents/birds-sounds/src/da...,Acropternis orthonyx,143668


In [3]:
# Cargar el ResNet152V2 (arquitectura del proyecto + pesos entrenados) y el label encoder
import pickle
import tensorflow as tf
from model_trainer import ModelTrainer
from image_preprocessor import ImagePreprocessor, ImagePreprocessorConfig

with open(LE_PATH, "rb") as f:
    label_encoder = pickle.load(f)

cfg = ImagePreprocessorConfig()                       # img (128,256), 1 canal, label_column='label'
pre = ImagePreprocessor(cfg, label_encoder=label_encoder)

trainer = ModelTrainer(model_name="ResNet152V2", n_classes=667,
                       weights=None, model_dir=os.path.join(ROOT, "notebooks/models"))
model = trainer.create_model()
# Al cargar, Keras avisa que omite el estado del optimizer Adam: es esperado e
# inofensivo para inferencia (solo importan los pesos del modelo).
model.load_weights(WEIGHTS)
print("ResNet152V2 cargado |", len(label_encoder.classes_), "clases")

/Users/camcortes/Documents/birds-sounds/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ResNet152V2 cargado | 667 clases


/Users/camcortes/Documents/birds-sounds/.venv/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 368 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [ ]:
# Inferencia por chunk (mismo preprocesamiento que la validación del proyecto) y agregación por grabación
ds = pre.create_validation_dataset(chunks_df)         # orden preservado (shuffle=False)
logits = model.predict(ds, verbose=1)
probs = tf.nn.softmax(logits, axis=1).numpy()

chunks_df["_i"] = range(len(chunks_df))
rows = []
for rec, g in chunks_df.groupby("recording_id"):
    mean_probs = probs[g._i.values].mean(axis=0)      # media de probabilidades sobre los chunks
    k = int(mean_probs.argmax())
    rows.append({
        "recording_id": rec,
        "species_true": g.label.iloc[0],
        "species_pred_resnet": label_encoder.inverse_transform([k])[0],
        "confidence_resnet": float(mean_probs[k]),
    })
resnet = pd.DataFrame(rows)
resnet.to_csv(os.path.join(DATA_DIR, "resnet_predictions.csv"), index=False)
print("predicciones ResNet recording-level:", len(resnet))
resnet.head()

In [ ]:
# Emparejar ambos modelos por grabación
comp = resnet.merge(
    bird_valid[["recording_id", "species_pred", "confidence"]].rename(
        columns={"species_pred": "species_pred_birdnet", "confidence": "confidence_birdnet"}),
    on="recording_id", how="inner")
comp["correct_resnet"]  = comp.species_pred_resnet  == comp.species_true
comp["correct_birdnet"] = comp.species_pred_birdnet == comp.species_true

# Anotar familia y género, y guardar la tabla pareada completa (trazabilidad para el NB22)
gf = bu.get_genus_family_map()
comp["family"] = comp.species_true.map(lambda s: bu.family_of(s, gf))
comp["genus"]  = comp.species_true.map(lambda s: str(s).split()[0])
comp.to_csv(os.path.join(DATA_DIR, "birdnet_vs_resnet_paired.csv"), index=False)

print("grabaciones comparadas (pareadas):", len(comp))
comp.head()

In [ ]:
# Métricas comparativas (mismas grabaciones y mismas especies)
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

def compute_metrics(y_true, y_pred):
    return {
        "accuracy":        accuracy_score(y_true, y_pred),
        "f1_macro":        f1_score(y_true, y_pred, average="macro", zero_division=0),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro":    recall_score(y_true, y_pred, average="macro", zero_division=0),
    }

summary = pd.DataFrame({
    "ResNet152V2":  compute_metrics(comp.species_true, comp.species_pred_resnet),
    "BirdNET_V2.4": compute_metrics(comp.species_true, comp.species_pred_birdnet),
}).T
summary["n_recordings"] = len(comp)
summary.to_csv(os.path.join(DATA_DIR, "birdnet_vs_resnet_summary.csv"))
print(summary.round(4).to_string())

              accuracy  f1_macro  precision_macro  recall_macro  n_recordings
ResNet152V2     0.8845    0.8823           0.8920        0.8806         11695
BirdNET_V2.4    0.9027    0.8995           0.9122        0.8992         11695


In [ ]:
# Desempeño por familia filogenética (accuracy de cada modelo)
by_family = (comp.groupby("family")
             .agg(n=("recording_id", "size"),
                  acc_resnet=("correct_resnet", "mean"),
                  acc_birdnet=("correct_birdnet", "mean"))
             .sort_values("n", ascending=False))
by_family["delta_resnet_minus_birdnet"] = by_family.acc_resnet - by_family.acc_birdnet
by_family.to_csv(os.path.join(DATA_DIR, "birdnet_vs_resnet_by_family.csv"))
print(by_family.head(15).round(3).to_string())

                   n  acc_resnet  acc_birdnet  delta_resnet_minus_birdnet
family                                                                   
Tyrannidae      2356       0.877        0.934                      -0.057
Thamnophilidae  1627       0.899        0.922                      -0.023
Thraupidae      1483       0.864        0.813                       0.051
Furnariidae     1378       0.874        0.913                      -0.039
Troglodytidae    596       0.874        0.864                       0.010
Icteridae        511       0.845        0.840                       0.006
Passerellidae    504       0.893        0.893                       0.000
Vireonidae       398       0.907        0.902                       0.005
Grallaridae      384       0.909        0.951                      -0.042
Rhinocryptidae   296       0.926        0.970                      -0.044
Fringillidae     275       0.891        0.778                       0.113
Hirundinidae     230       0.913      

In [ ]:
# Test de McNemar: ¿la diferencia de aciertos pareados es significativa?
from statsmodels.stats.contingency_tables import mcnemar

both  = int(( comp.correct_resnet &  comp.correct_birdnet).sum())
r_only = int(( comp.correct_resnet & ~comp.correct_birdnet).sum())  # solo ResNet acierta
b_only = int((~comp.correct_resnet &  comp.correct_birdnet).sum())  # solo BirdNET acierta
none  = int((~comp.correct_resnet & ~comp.correct_birdnet).sum())

table = [[both, r_only], [b_only, none]]
res = mcnemar(table, exact=False, correction=True)
print(f"Tabla de contingencia [[both, resnet_only],[birdnet_only, none]] = {table}")
print(f"McNemar chi2 = {res.statistic:.4f} | p-value = {res.pvalue:.4g}")
print(f"ResNet acierta y BirdNET no: {r_only} | BirdNET acierta y ResNet no: {b_only}")

Tabla de contingencia [[both, resnet_only],[birdnet_only, none]] = [[9506, 838], [1051, 300]]
McNemar chi2 = 23.7925 | p-value = 1.073e-06
ResNet acierta y BirdNET no: 838 | BirdNET acierta y ResNet no: 1051


In [ ]:
import matplotlib.pyplot as plt

mcols  = ["accuracy", "f1_macro", "precision_macro", "recall_macro"]
xlabs  = ["Accuracy", "F1 macro", "Precision\nmacro", "Recall\nmacro"]
xr = np.arange(len(mcols)); w = 0.38
fig, ax = plt.subplots(figsize=(8, 4.5))
b1 = ax.bar(xr - w/2, summary.loc["ResNet152V2", mcols].values,  w, label="ResNet152V2",
            color=vs.MODEL_COLORS["ResNet152V2"], edgecolor="white", linewidth=0.5, alpha=0.88)
b2 = ax.bar(xr + w/2, summary.loc["BirdNET_V2.4", mcols].values, w, label="BirdNET V2.4",
            color=vs.MODEL_COLORS["BirdNET_V2.4"], edgecolor="white", linewidth=0.5, alpha=0.88)
ax.set_xticks(xr); ax.set_xticklabels(xlabs)
ax.set_ylim(0.82, 1.04); ax.set_ylabel("Score")
ax.yaxis.grid(True, alpha=0.3, linewidth=0.5, zorder=0)
_n = f"{len(comp):,}".replace(",", " ")
ax.set_title(f"ResNet152V2 vs BirdNET V2.4  ·  Recording-level  (n = {_n})")
ax.legend(); vs.despine(ax)
for i, c in enumerate(mcols):
    vr = summary.loc["ResNet152V2", c]; vb = summary.loc["BirdNET_V2.4", c]
    ax.text(i - w/2, vr + .008, f"{vr:.3f}", ha="center", fontsize=8, fontweight="semibold")
    ax.text(i + w/2, vb + .008, f"{vb:.3f}", ha="center", fontsize=8, fontweight="semibold")
fig.tight_layout()
vs.savefig_dual(fig, "birdnet_vs_resnet_metrics")
plt.show()

In [ ]:
# Accuracy by family (top 20 families by recording count)
topf = by_family.head(20)
xr = np.arange(len(topf)); w = 0.38
fig, ax = plt.subplots(figsize=(13, 5))
b1 = ax.bar(xr - w/2, topf.acc_resnet.values,  w, label="ResNet152V2",
            color=vs.MODEL_COLORS["ResNet152V2"], edgecolor="white", linewidth=0.5, alpha=0.88, zorder=3)
b2 = ax.bar(xr + w/2, topf.acc_birdnet.values, w, label="BirdNET V2.4",
            color=vs.MODEL_COLORS["BirdNET_V2.4"], edgecolor="white", linewidth=0.5, alpha=0.88, zorder=3)
ax.set_xticks(xr); ax.set_xticklabels(topf.index, rotation=45, ha="right", fontsize=9, style="italic")
ax.set_ylim(0.55, 1.08); ax.set_ylabel("Accuracy")
ax.yaxis.grid(True, alpha=0.3, linewidth=0.5, zorder=0)
ax.set_title("Accuracy by Family  ·  Top 20 by Recording Count",
             fontsize=13, fontweight="bold", pad=10)
ax.legend(); vs.despine(ax)
fig.tight_layout()
vs.savefig_dual(fig, "birdnet_vs_resnet_by_family")
plt.show()

In [ ]:
# Resumen final
print("="*60)
print("COMPARACIÓN ResNet152V2 vs BirdNET V2.4 (recording-level)")
print("="*60)
print(summary.round(4).to_string())
print(f"\nGrabaciones pareadas: {len(comp)}  |  especies: {comp.species_true.nunique()}")
print(f"McNemar p-value: {res.pvalue:.4g}")
print("\nArchivos generados:")
for fn in ["resnet_predictions.csv", "birdnet_vs_resnet_summary.csv", "birdnet_vs_resnet_by_family.csv"]:
    print("  src/data/" + fn)
for fn in ["birdnet_vs_resnet_metrics", "birdnet_vs_resnet_by_family"]:
    print("  fig/" + fn + ".png / .pdf")

COMPARACIÓN ResNet152V2 vs BirdNET V2.4 (recording-level)
              accuracy  f1_macro  precision_macro  recall_macro  n_recordings
ResNet152V2     0.8845    0.8823           0.8920        0.8806         11695
BirdNET_V2.4    0.9027    0.8995           0.9122        0.8992         11695

Grabaciones pareadas: 11695  |  especies: 664
McNemar p-value: 1.073e-06

Archivos generados:
  src/data/resnet_predictions.csv
  src/data/birdnet_vs_resnet_summary.csv
  src/data/birdnet_vs_resnet_by_family.csv
  fig/birdnet_vs_resnet_metrics.png / .pdf
  fig/birdnet_vs_resnet_by_family.png / .pdf
